# 第02回 AIの歴史と簡易ELIZA：値とコードを一段ずつ読む

簡易ELIZAの応答を、入力文字列・規則の順序・対話回数から説明します。意味理解や学習を実装したものではありません。

**開始：** 既存venvのカーネルを選び、初回は変更せず「カーネル再起動→すべて実行」。次に一節ずつ、予想してからShift+Enterで実行します。同梱Pythonのimportは不要で、外部通信も行いません。

**読み方：** 関数の定義は計算方法に名前を付けるセルです。続く短い実行セルが入力を渡します。表・図の表示コードは折り畳み指定です。表示は丸めますが計算には元の値を使います。

**完了の目印：** 母が心配ですでは母の規則が先に一致し、既定cycleでは6回目にオウム返し。固定20応答は従来どおり。


In [ ]:
import sys
print('使用中のPython:',sys.executable)
print('版:',sys.version.split()[0])


In [ ]:
import argparse


In [ ]:
import re


In [ ]:
LIMIT = 20


In [ ]:
CYCLE = 5


In [ ]:
GREETING = "私はDoctor、お話を伺います"


In [ ]:
FAREWELL = "それではそろそろ終了しましょう。おつかれさまでした。"


In [ ]:
RULES = [
    ("先生", "私のことでなくあなたのことを話しましょう"),
    ("母", "あなたのお母さんについて話してください"),
    ("父", "あなたのお父さんについて話してください"),
    ("意見", "私の意見を聞きたいのですか？"),
    ("が心配です", "{question}"),
]


In [ ]:
DEMO_INPUTS = [
    "先生に相談したいです", "母が心配です", "父について話します",
    "あなたの意見を聞きたいです", "試験が心配です", "今日は晴れです",
    "こんにちは", "先生と母について", "父と母について", "母校の話です",
    "試験が心配です", "意見と父について", "授業が心配です",
    "父が心配です", "母が心配です", "先生について", "意見があります",
    "締切が心配です", "こんにちは", "最後の入力です",
]


### 図表の準備（変更不要）

表示用の補助コードです。HTML/SVGを覚える必要はありません。折り畳み表示はNotebook環境によって異なります。


In [ ]:
from html import escape


In [ ]:
from IPython.display import HTML, display


In [ ]:
def learning_format(value):
    if isinstance(value, float):
        return f'{value:.6g}'
    if isinstance(value, (list, tuple)):
        return '['+', '.join(learning_format(item) for item in value)+']'
    if isinstance(value, dict):
        return '{'+', '.join(str(key)+': '+learning_format(item) for key,item in value.items())+'}'
    return str(value)


In [ ]:
def learning_table(title, columns, rows):
    header = ''.join('<th>'+escape(str(c))+'</th>' for c in columns)
    body = ''
    for row in rows:
        body += '<tr>'+''.join('<td>'+escape(learning_format(x))+'</td>' for x in row)+'</tr>'
    return '<div class="scroll"><table><caption>'+escape(title)+'</caption><tr>'+header+'</tr>'+body+'</table></div>'


In [ ]:
def learning_grid(title, matrix):
    rows = [(i,*row) for i,row in enumerate(matrix)]
    return learning_table(title,['行↓ 列→',*range(len(matrix[0]))],rows)


In [ ]:
def learning_plot(title, points):
    # 実計算の値だけを座標へ変換。点の間の値は補間して表示しない。
    xs, ys = [p[0] for p in points], [p[1] for p in points]
    xmin,xmax,ymin,ymax = min(xs),max(xs),min(0,min(ys)),max(ys)
    sx = lambda x: 100+(x-xmin)/max(1e-12,xmax-xmin)*500
    sy = lambda y: 240-(y-ymin)/max(1e-12,ymax-ymin)*190
    marks = ''
    for value in (ymin,(ymin+ymax)/2,ymax):
        marks += f'<path d="M100 {sy(value)} H600" stroke="#aaa"/><text x="0" y="{sy(value)+5}">{value:.4g}</text>'
    for x,y in points:
        marks += f'<circle cx="{sx(x)}" cy="{sy(y)}" r="4" fill="#155e75"><title>{x}: {y}</title></circle>'
    marks += f'<text x="100" y="275">{xmin}</text><text x="580" y="275">{xmax}</text>'
    return '<svg viewBox="0 0 700 310" role="img" aria-label="'+escape(title)+'"><text x="100" y="22">'+escape(title)+'</text>'+marks+'</svg>'


In [ ]:
def learning_show(report, reset=None):
    style = '<style>.learning{font:17px/1.65 sans-serif;color:#172b42;background:white;padding:12px}.learning .scroll{overflow-x:auto}.learning table{border-collapse:collapse;margin:14px 0}.learning td,.learning th{border:1px solid #8394a5;padding:6px 12px;white-space:normal}.learning caption{text-align:left;font-weight:bold}.learning svg{max-width:100%;height:auto}.learning .boards{display:flex;gap:20px;flex-wrap:wrap}.learning .result{border-left:4px solid #155e75;padding:10px}</style>'
    content = '<h3>'+escape(report['title'])+'</h3><p>入力：'+escape(learning_format(report['input']))+'</p>'
    content += learning_table('使う値 → 操作 → 結果',report['columns'],report['rows'])
    for title,columns,rows in report.get('tables',[]):
        content += '<details><summary>詳しい計算：'+escape(title)+'</summary>'+learning_table(title,columns,rows)+'</details>'
    content += '<div class="boards">'+''.join(learning_grid(title,values) for title,values in report.get('matrices',[]))+'</div>'
    for title,points in report.get('plots',[]):
        content += learning_plot(title,points)
    for label in ('result','comparison','explanation','reset'):
        name={'result':'結果','comparison':'一つ変えた比較','explanation':'読み取り方','reset':'戻す'}[label]
        content += '<p class="result">'+name+'：'+escape(reset if label=='reset' and reset is not None else learning_format(report[label]))+'</p>'
    display(HTML(style+'<div class="learning">'+content+'</div>'))


## 1. 文字列と規則の一致

**今いる段階：** 入力→照合→停止→応答

**予想：** どの規則が先に一致するか

**操作：** example_textを'母が心配です'から'試験が心配です'へ変更し、同じセルを実行します。

**結果の目印：** 母の規則で停止し、心配の規則まで進まない

**戻す：** example_text='母が心配です'に戻して再実行します。

**説明する：** 未確認と不一致の違い


### 計算の定義：`Doctor`

応答の状態を持つ。rules・cycle・limit を変更して比較できる。


In [ ]:
class Doctor:
    """応答の状態を持つ。rules・cycle・limit を変更して比較できる。"""

    def __init__(self, rules=None, cycle=CYCLE, limit=LIMIT):
        if cycle < 1 or limit < 1:
            raise ValueError("cycle と limit は1以上にしてください")
        self.rules = list(RULES if rules is None else rules)
        self.cycle = cycle
        self.limit = limit
        self.count = 0
        self.turns = 0
        self.last_rule = ""

    @property
    def finished(self):
        return self.turns >= self.limit

    def respond(self, text):
        if self.finished:
            raise StopIteration("対話は終了しました")
        self.last_checks = []
        # 昨年度と同じ更新順。既定では6・11・16回目にオウム返しする。
        if self.count >= self.cycle:
            response = text + " 、ですか・・・"
            self.count = 0
            self.last_rule = "オウム返し"
        else:
            response = "続けてください"
            self.last_rule = "既定応答"
            for pattern, template in self.rules:
                matched = bool(re.search(pattern, text))
                self.last_checks.append((pattern, matched))
                if matched:
                    response = template.format(
                        question=text.replace("が心配です", "は心配ですか？")
                    )
                    self.last_rule = pattern
                    break
        self.count += 1
        self.turns += 1
        return response


### 計算の定義：`lesson_rule`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_rule(text='母が心配です'):
    doctor = Doctor()
    response = doctor.respond(text)
    rows = [(i+1,pattern,matched,'採用して停止' if matched else '次の規則へ') for i,(pattern,matched) in enumerate(doctor.last_checks)]
    return {'title':'入力→規則を上から照合→応答','question':'心配という語があれば必ず質問へ置き換わるか',
            'input':text,'columns':['順番','調べた規則','一致','次の処理'],'rows':rows,
            'result':{'採用規則':doctor.last_rule,'応答':response},'comparison':'母が心配です→試験が心配ですで、採用される規則が変わる。',
            'explanation':'最初に一致した規則だけを採用する。未確認の後続規則を不一致と扱わない。意味理解や学習をしているわけではない。',
            'reset':'text="母が心配です"で再実行。新しいDoctorなので対話回数も0から。'}


In [ ]:
example_text = '母が心配です'
program_observation_rule = lesson_rule(example_text)
learning_show(program_observation_rule, reset="example_text='母が心配です'に戻して再実行します。")


## 2. 規則の並べ替え

**今いる段階：** 同じ入力→異なる順序→応答

**予想：** 複数の語があると何を採用するか

**操作：** example_reverseをFalseからTrueへ変更し、同じセルを実行します。

**結果の目印：** 通常は先生、逆順は母

**戻す：** example_reverse=Falseに戻して再実行します。

**説明する：** 規則の変更と機械学習の違い


### 計算の定義：`lesson_order`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_order(reverse=False):
    rules = list(RULES)
    if reverse:
        rules.reverse()
    doctor = Doctor(rules=rules)
    text = '先生と母について'
    response = doctor.respond(text)
    return {'title':'同じ入力でも規則の順序で変わる','question':'先生と母の両方があると何を選ぶか',
            'input':f'入力={text}、規則を逆順={reverse}',
            'columns':['確認順','規則','一致'],'rows':[(i+1,p,m) for i,(p,m) in enumerate(doctor.last_checks)],
            'tables':[('今回の規則順',['順','規則','応答ひな型'],[(i+1,*rule) for i,rule in enumerate(rules)])],
            'result':response,'comparison':'通常順は先生、逆順では母が先に一致する。',
            'explanation':'応答の変化は規則を人が並べ替えた結果。データから重みを学習した結果ではない。',
            'reset':'reverse=Falseで再実行。RULES自体は変更しない。'}


In [ ]:
example_reverse = False
program_observation_order = lesson_order(example_reverse)
learning_show(program_observation_order, reset='example_reverse=Falseに戻して再実行します。')


## 3. 回数を持つ応答

**今いる段階：** 状態→判定→応答→新しい状態

**予想：** countはいつ増えるか

**操作：** example_cycleを5から2へ変更し、同じセルを実行します。

**結果の目印：** 判定前と更新後を8件の表で追える

**戻す：** example_cycle=5に戻して再実行します。

**説明する：** countとturnsの違い


### 計算の定義：`lesson_cycle`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_cycle(cycle=5):
    doctor = Doctor(cycle=cycle)
    rows = []
    for text in DEMO_INPUTS[:8]:
        before = doctor.count
        response = doctor.respond(text)
        rows.append((doctor.turns,text,before,doctor.last_rule,response,doctor.count))
    return {'title':'対話回数という状態を追う','question':'同じ入力でも直前までの回数で変わるか',
            'input':f'固定入力の先頭8件、cycle={cycle}。limit=20。',
            'columns':['対話番号','入力','判定前count','採用規則','応答','更新後count'],'rows':rows,
            'result':f'8件後のcount={doctor.count}、turns={doctor.turns}',
            'comparison':'cycle=5では6回目、cycle=2では3回目からオウム返しになる。',
            'explanation':'先にcount≥cycleを判定し、オウム返しでは0へ戻してから1を足す。countと全対話数turnsは別。',
            'reset':'cycle=5で再実行。新しいDoctorから同じ8件を処理する。'}


In [ ]:
example_cycle = 5
program_observation_cycle = lesson_cycle(example_cycle)
learning_show(program_observation_cycle, reset='example_cycle=5に戻して再実行します。')


## 基準結果と既存の変更実験

ここまでの図表を、従来の基準出力と照合します。以下の実験も入力のコピーを使います。定義を変更した場合は再起動して全セル実行してください。


### 計算の定義：`demo_responses`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def demo_responses(inputs=DEMO_INPUTS):
    doctor = Doctor()
    responses = []
    for text in inputs:
        if doctor.finished:
            break
        responses.append(doctor.respond(text))
    return responses


In [ ]:
import sys
print("実行中のPython:", sys.executable)
print("Pythonの版:", sys.version.split()[0])


## 1. 規則と、対話の状態

`RULES` を上から調べ、最初に一致した規則を一つ使います。「が心配です」の規則は文字列を「は心配ですか？」へ置換します。該当なしは「続けてください」です。

`count` はオウム返しまでの状態、`turns` は全体の応答回数です。初期設定は `CYCLE=5`、`LIMIT=20`。昨年度の更新順を維持しているため、6・11・16回目にはキーワード規則より先にオウム返しします。

`Doctor()` で新しい状態を作れます。比較する際は同じ状態から始めましょう。以下には単体実行に必要な処理がすべてあります。


## 2. 同じ入力20件で、基準の応答を確認する

入力一覧は上の `DEMO_INPUTS` にあります。各行に応答と使った規則を表示します。CLIの `python doctor.py --demo` と同じ応答列になります。

**観察点**：「母が心配です」は「母」が優先される。6・11・16回目はオウム返し。「母校」も文字列「母」に一致する。20回で終了する。乱数はなく、文字列は完全一致で照合できます。


In [ ]:
baseline_responses = demo_responses()
doctor = Doctor()
for number, text in enumerate(DEMO_INPUTS, 1):
    response = doctor.respond(text)
    print(f"{number:02d} あなた> {text}")
    print(f"   Dr> {response}  [規則: {doctor.last_rule}]")
print("応答回数:", doctor.turns, "終了:", doctor.finished)


## 3. 入力だけ変える

同じ規則を使い、それぞれ新しい状態から始めます。次の三つの応答を比べてください。

- 「母が心配です」と「試験が心配です」は、どの規則が違うでしょうか。
- 「母校の話です」の応答は、入力の意味に合っているでしょうか。
- 一致した文字列から、その語の意味まで処理したと言えるでしょうか。

`inputs_to_try` の内容を変え、実行し直せます。


In [ ]:
inputs_to_try = ["母が心配です", "試験が心配です", "母校の話です"]
for text in inputs_to_try:
    doctor = Doctor()
    response = doctor.respond(text)
    print(text, "→", response, "[規則:", doctor.last_rule, "]")


## 4. 規則の順序だけ変える

「が心配です」を先頭へ移します。入力は同じ「母が心配です」です。変更前・変更後の両方を、新しい状態から試します。

**想定結果**：変更前は「あなたのお母さんについて話してください」、変更後は「母は心配ですか？」。複数の規則が一致しても、最初の一つだけが使われます。


In [ ]:
changed_rules = [RULES[-1]] + RULES[:-1]
text = "母が心配です"
for label, rules in [("変更前", RULES), ("変更後", changed_rules)]:
    doctor = Doctor(rules=rules)
    response = doctor.respond(text)
    print(label, response, "[規則:", doctor.last_rule, "]")


## 5. キーワードと応答を追加する

先頭に「締切」の規則を加えます。`custom_rules` のキーワードや応答文を変えて試せます。

**想定結果**：「締切が心配です」に対し、変更前は「締切は心配ですか？」、追加後は「締切までに何を進めますか？」。

これは人が規則を書き換えた結果です。プログラムが対話例から規則を自動で学んだわけではありません。


In [ ]:
custom_rules = [("締切", "締切までに何を進めますか？")] + RULES
text = "締切が心配です"
print("変更前:", Doctor().respond(text))
print("追加後:", Doctor(rules=custom_rules).respond(text))


## 6. 状態の影響を確認する

通常の規則と、短い周期 `cycle=2` を比較します。各版は新しい状態から始め、同じ「先生」を6回入力します。

**想定結果**：既定では6回目、短い周期では3・5回目がオウム返しです。同じ入力と規則でも、対話状態が違えば応答が変わります。回数の設定を変更することも、自動学習ではありません。


In [ ]:
for label, cycle in [("既定", 5), ("短い周期", 2)]:
    doctor = Doctor(cycle=cycle, limit=6)
    print(label)
    for number in range(1, 7):
        print(number, doctor.respond("先生"), "[規則:", doctor.last_rule, "]")


## 観察のまとめ

1. どの入力に、どの規則が一致したかを説明する。
2. 規則を並べ替えた場合は、入力と開始状態をそろえて比較する。
3. 文字列の一致、自然な返答、意味の理解を同じものとして扱わない。
4. 規則を編集する操作と、データを使って重みなどを調整する学習を区別する。

提出・採点の条件はこの教材で追加しません。プログラムの検証は章内で `make check`、リポジトリのルートで `make check-programs` を使います。実行済みNotebook・検証結果は `programs/outputs/` に分離し、配布するこの原稿の出力セルは空に保ちます。


## 振り返りと復帰

入力・行った処理・結果・変更前との差を、自分の言葉で説明してください。観察セルは毎回基準のコピーから計算するため、同じセルを再実行しても前の実験の続きにはなりません。NameErrorや実行順が不明な場合はカーネル再起動→全セル実行。元の配布ファイルを残し、変更版は別名保存します。

Python版は`python doctor.py --explain all`で同じ計算表を表示します。各テーマは`--explain rule`のように指定できます。従来のコマンドと固定期待値も維持しています。
